In [ ]:
import json
import pandas as pd
from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score
)

# =====================================================
# 1. CHARGEMENT DU CORPUS (MultiNLI)
# =====================================================

DATA_PATH = "multinli_1.0/multinli_1.0_train.jsonl"

data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        if item["gold_label"] != "-":
            data.append(item)

data = data[:100]

df = pd.DataFrame(data)[["sentence1", "sentence2", "gold_label"]]

print("Distribution des labels :")
print(df["gold_label"].value_counts(normalize=True))
print("-" * 50)

Distribution des labels :
gold_label
contradiction    0.375
entailment       0.343
neutral          0.282
Name: proportion, dtype: float64
--------------------------------------------------


In [3]:
# =====================================================
# 2. SPLIT TRAIN / DEV / TEST
# =====================================================

train_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df["gold_label"]
)

train_df, dev_df = train_test_split(
    train_df, test_size=0.176, random_state=42, stratify=train_df["gold_label"]
)

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print("-" * 50)

Train: 700 | Dev: 150 | Test: 150
--------------------------------------------------


In [4]:
# =====================================================
# 3. CHARGEMENT DU MODELE NLI
# =====================================================

nli = pipeline(
    task="text-classification",
    model="roberta-large-mnli",
    return_all_scores=True
)

Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
c:\Users\hagop\Desktop\M2_ISD\Stat\.venv\Lib\site-packages\transformers\pipelines\text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=Fa

In [11]:
# =====================================================
# 4. PREDICTION AVEC SEUILS
# =====================================================

ENTAILMENT_THRESHOLD = 0.8
CONTRADICTION_THRESHOLD = 0.8

def predict_nli(sentence1, sentence2):
    output = nli({"text": sentence1, "text_pair": sentence2})
    # output is a list of dicts; iterate directly
    scores = {x['label'].lower(): float(x['score']) for x in output}

    if scores["entailment"] >= ENTAILMENT_THRESHOLD:
        decision = "VRAI"
        label = "entailment"
        confidence = scores["entailment"]

    elif scores["contradiction"] >= CONTRADICTION_THRESHOLD:
        decision = "FAUX"
        label = "contradiction"
        confidence = scores["contradiction"]

    else:
        decision = "A_VERIFIER"
        label = "neutral"
        confidence = scores["neutral"]

    return label, decision, confidence

In [12]:
# =====================================================
# 5. EVALUATION SUR LE TEST
# =====================================================

predicted_labels = []
decisions = []
confidences = []

for _, row in test_df.iterrows():
    label, decision, conf = predict_nli(row["sentence1"], row["sentence2"])
    predicted_labels.append(label)
    decisions.append(decision)
    confidences.append(conf)

test_df = test_df.copy()
test_df["predicted_label"] = predicted_labels
test_df["decision"] = decisions
test_df["confidence"] = confidences

In [13]:
# =====================================================
# 6. METRIQUES
# =====================================================

accuracy = accuracy_score(test_df["gold_label"], test_df["predicted_label"])
kappa = cohen_kappa_score(test_df["gold_label"], test_df["predicted_label"])
conf_matrix = confusion_matrix(test_df["gold_label"], test_df["predicted_label"])

print("Accuracy :", round(accuracy * 100, 2), "%")
print("Cohen Kappa :", round(kappa, 3))
print("\nMatrice de confusion :")
print(conf_matrix)

print("\nRapport de classification :")
print(classification_report(
    test_df["gold_label"],
    test_df["predicted_label"]
))

Accuracy : 96.0 %
Cohen Kappa : 0.94

Matrice de confusion :
[[53  0  3]
 [ 0 49  3]
 [ 0  0 42]]

Rapport de classification :
               precision    recall  f1-score   support

contradiction       1.00      0.95      0.97        56
   entailment       1.00      0.94      0.97        52
      neutral       0.88      1.00      0.93        42

     accuracy                           0.96       150
    macro avg       0.96      0.96      0.96       150
 weighted avg       0.96      0.96      0.96       150



In [14]:
# =====================================================
# 7. ANALYSE SIMPLE DES ERREURS
# =====================================================

errors = test_df[test_df["gold_label"] != test_df["predicted_label"]]
print(f"Nombre d'erreurs : {len(errors)} / {len(test_df)}")

print("\nExemples d'erreurs :")
print(errors[[
    "sentence1",
    "sentence2",
    "gold_label",
    "predicted_label",
    "confidence"
]].head(5))

Nombre d'erreurs : 6 / 150

Exemples d'erreurs :
                                             sentence1  \
852  Jon placed a hand on San'doro's shoulder and s...   
763  You won't envy the men and women you see worki...   
967  But these are the trivia of what he left me an...   
338  they use the the injection thing or whatever i...   
873                        i wonder how long that last   

                                             sentence2     gold_label  \
852         San'doro picked up Jon so he could stand.   contradiction   
763  The people who labor outside are optimistic an...     entailment   
967  But these are the answers of what he left me a...  contradiction   
338                         They use lethal injection.     entailment   
873                        I don't think it will last.  contradiction   

    predicted_label  confidence  
852         neutral    0.278051  
763         neutral    0.212615  
967         neutral    0.084255  
338         neutral    0.60

In [18]:
sentence1 = "Yanis is smart"
sentence2 = "Yanis is stupid"

label, decision, confidence = predict_nli(sentence1, sentence2)
print(f"Phrase 1 : {sentence1}")
print(f"Phrase 2 : {sentence2}")
print(f"Décision : {decision} (label: {label}, confiance: {round(confidence, 3)})")


Phrase 1 : Yanis is smart
Phrase 2 : Yanis is stupid
Décision : FAUX (label: contradiction, confiance: 0.999)
